In [ ]:
# Importing Required Module
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from tabulate import tabulate 
import pandas as pd 
import logging
import os 
from sqlalchemy.engine import Engine

# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

# Database Configuration
DB_USER = quote_plus(os.getenv("DB_USER"))
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))
DB_HOST = "localhost"
DB_PORT = "1433"
DB_NAME = "TestDB"
ODBC_DRIVER = ("ODBC Driver 18 for SQL Server")
TRUST_SERVER_CERTIFICATE = "yes"

# Validation
if not DB_USER :
    raise ValueError("DB_USER enverment variable not found")

if not DB_PASSWORD : 
    raise ValueError("DB_PASSWORD enverment variable not found")


# Database Connection
def create_sql_server_engine() -> Engine:
    """
    create or return a SQL Server SQLAlchemy engine 
    """
    connection_string = (
        f"mssql+pyodbc://{DB_USER}:{DB_PASSWORD}"
        f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
        f"?driver={quote_plus(ODBC_DRIVER)}"
        f"&TrustServerCertificate={TRUST_SERVER_CERTIFICATE}"
    )

    try:
        engine = create_engine(connection_string)
        with engine.connect():
            logger.info("Successfully connected to SQL Server.")
        return engine

    except Exception as error:
        logger.exception("Failed to create database connection.")
        raise error

# Query Execution
def read_data(query : str , engine : Engine) -> pd.DataFrame:
    """
    Execute query and return dataframe
    """

    try :
        with engine.connect() as connection:
            dataframe = pd.read_sql(query, connection)

        logger.info(
            "Query executed successfully. Row return %s",
            len(dataframe)
        )
        return dataframe

    except Exception as error:
        logger.exception("Query execution faild.")
        raise error 
# ------------------------------------------------------------------------------
# Main
# ------------------------------------------------------------------------------

def main() -> None:

    query = """
    SELECT TOP 10
        customer_id,
        title,
        first_name,
        last_name,
        full_name,
        gender
    FROM bronze.customers;
    """

    engine = create_sql_server_engine()

    df = read_data(query, engine)

    print(
        tabulate(
            df,
            headers="keys",
            tablefmt="simple_grid",
            showindex=False
        )
    )


if __name__ == "__main__":
        main()